In [1]:
from groq import Groq
from dotenv import load_dotenv
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [2]:
from pydantic import BaseModel

class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

In [3]:
CalendarEvent.model_json_schema()

{'properties': {'name': {'title': 'Name', 'type': 'string'},
  'date': {'title': 'Date', 'type': 'string'},
  'participants': {'items': {'type': 'string'},
   'title': 'Participants',
   'type': 'array'}},
 'required': ['name', 'date', 'participants'],
 'title': 'CalendarEvent',
 'type': 'object'}

In [8]:
import json

schema = CalendarEvent.model_json_schema()

response = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": f"Extract the event information. Return JSON only matching this schema: {schema}"},
        {"role": "user", "content": "Alice and Bob are going to a science fair on Friday."},
    ],
    response_format={"type": "json_object"}
)

raw = response.choices[0].message.content
print(raw)
event = CalendarEvent(**json.loads(raw))
print(event)

{
  "name": "science fair",
   "date": "Friday",
   "participants": [
      "Alice",
      "Bob"
   ]
}
name='science fair' date='Friday' participants=['Alice', 'Bob']
